In [ ]:
# !pip install pandas sentence-transformers transformers docling langchain langchain-text-splitters

import pandas as pd
from pathlib import Path
from google.colab import drive

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from docling.datamodel.document import DoclingDocument
from docling.chunking import HybridChunker, HierarchicalChunker
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
drive.mount('/content/drive')

JSON_DIR = Path("/content/drive/MyDrive/docling_pdfs/json")
EXPORT_DIR = Path("/content/drive/MyDrive/docling_pdfs/RAG_Export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-large"
print(f"⏳ Lade Modell: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = SentenceTransformer(MODEL_NAME)

In [ ]:
for i, json_path in enumerate(JSON_DIR.glob("*.json")):
  document = DoclingDocument.load_from_json(json_path)
  print(document.name)

In [ ]:
# Chunker initialisieren
# Methode 1: Docling Hybrid (Strikes Token-Limit 512)
hybrid_chunker = HybridChunker(tokenizer=tokenizer, max_tokens=512)

# Methode 2: Docling Hierarchical (Strukturbasiert)
hierarchical_chunker = HierarchicalChunker()

# Methode 3: LangChain Markdown Header
headers_to_split_on = [("#", "H1"), ("##", "H2"), ("###", "H3")]
md_header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# Methode 4: LangChain Recursive (1500 Zeichen entsprechen grob 400-500 Tokens)
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)

# Listen zum Sammeln der finalen Daten
data_hybrid = []
data_hierarchical = []
data_md_header = []
data_recursive = []

def embed_and_store(chunks, doc_id, target_list):
    if not chunks:
        return

    prefixed_texts = [f"passage: {text}" for text in chunks]

    embeddings = model.encode(prefixed_texts).tolist()

    for i, (text, emb) in enumerate(zip(chunks, embeddings)):
        if not text.strip():
            continue

        target_list.append({
            "doc_id": doc_id,
            "chunk_index": i,
            "content": text,
            "token_count": len(tokenizer.encode(text)),
            "embedding": emb
        })

existing_json_files = 0
not_existing_json_files = 0

print("🚀 Starte Chunking und Embedding-Generierung...")

for i, json_path in enumerate(JSON_DIR.glob("*.json")):
    existing_json_files += 1

    document = DoclingDocument.load_from_json(json_path)

    md_text = document.export_to_markdown()

    # --- CHUNKING ---
    # 1. Hybrid (Docling)
    chunks_hybrid = [c.text for c in hybrid_chunker.chunk(document)]

    # 2. Hierarchical (Docling)
    chunks_hierarchical = [c.text for c in hierarchical_chunker.chunk(document)]

    # 3. MD Header + Recursive
    # Schritt A: Nach Überschriften trennen
    md_splits = md_header_splitter.split_text(md_text)
    # Schritt B: Zur Sicherheit nochmal den Recursive Splitter drüberlaufen lassen
    safe_md_splits = recursive_splitter.split_documents(md_splits)
    chunks_md_header = [doc.page_content for doc in safe_md_splits]

    # 4. Nur Recursive Splitter
    chunks_recursive = recursive_splitter.split_text(md_text)

    # --- EMBEDDINGS ERSTELLEN & SAMMELN ---
    embed_and_store(chunks_hybrid, document.name, data_hybrid)
    embed_and_store(chunks_hierarchical, document.name, data_hierarchical)
    embed_and_store(chunks_md_header, document.name, data_md_header)
    embed_and_store(chunks_recursive, document.name, data_recursive)

    print(f"✅ {document.name} verarbeitet.")

In [ ]:
print(f"\nFertig! {existing_json_files} Dokumente verarbeitet.")

print("💾 Speichere Dateien auf Google Drive...")

pd.DataFrame(data_hybrid).to_parquet(EXPORT_DIR / "embeddings_docling_hybrid.parquet")
pd.DataFrame(data_hierarchical).to_parquet(EXPORT_DIR / "embeddings_docling_hierarchical.parquet")
pd.DataFrame(data_md_header).to_parquet(EXPORT_DIR / "embeddings_langchain_mdheader.parquet")
pd.DataFrame(data_recursive).to_parquet(EXPORT_DIR / "embeddings_langchain_recursive.parquet")

print(f"🎉 Alle 4 Parquet-Dateien liegen jetzt in: {EXPORT_DIR}")